# Aula 2 — Rede Neural From Scratch (NumPy)

Notebook de acompanhamento da Aula 2 — **Tópicos Avançados em IA · UFABC**  
Adaptado de MIT 15.773 (Farias & Ramakrishnan, 2024)

Implementamos **tudo manualmente com NumPy**: forward pass, função de perda, backpropagation via regra da cadeia, e atualização de pesos. Nenhum framework de deep learning é utilizado.

---

## Índice

1. [Configuração e Dataset](#1-configuração-e-dataset)
2. [Funções de Ativação e Derivadas](#2-funções-de-ativação-e-derivadas)
3. [Inicialização dos Pesos](#3-inicialização-dos-pesos)
4. [Forward Pass](#4-forward-pass)
5. [Função de Perda (Binary Cross-Entropy)](#5-função-de-perda-binary-cross-entropy)
6. [Backpropagation](#6-backpropagation)
7. [Laço de Treino (Gradient Descent)](#7-laço-de-treino-gradient-descent)
8. [Avaliação](#8-avaliação)
9. [Visualizando o Grafo Computacional](#9-visualizando-o-grafo-computacional)

## 1. Configuração e Dataset

In [ ]:
!pip install -q ucimlrepo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

np.random.seed(42)

In [ ]:
from ucimlrepo import fetch_ucirepo

heart  = fetch_ucirepo(id=45)
X_raw  = heart.data.features.fillna(heart.data.features.median(numeric_only=True))
y_raw  = (heart.data.targets.values.ravel() > 0).astype(np.float64)

cat_cols = ['cp', 'restecg', 'slope', 'thal']
X_df = pd.get_dummies(X_raw, columns=cat_cols, drop_first=False)

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_df.values, y_raw, test_size=0.30,
                                             random_state=42, stratify=y_raw)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50,
                                             random_state=42, stratify=y_tmp)

scaler = StandardScaler()
X_tr  = scaler.fit_transform(X_tr)
X_val = scaler.transform(X_val)
X_te  = scaler.transform(X_te)

# Reshape targets para colunas (n, 1)
y_tr  = y_tr.reshape(-1, 1)
y_val = y_val.reshape(-1, 1)
y_te  = y_te.reshape(-1, 1)

n_in = X_tr.shape[1]
print(f"Treino: {X_tr.shape}  |  Val: {X_val.shape}  |  Teste: {X_te.shape}")
print(f"Entradas: {n_in}")

## 2. Funções de Ativação e Derivadas

O backprop precisa das **derivadas** dessas funções para calcular os gradientes locais no grafo computacional.

| Função | Fórmula | Derivada |
|---|---|---|
| Sigmoide | $\sigma(z) = \frac{1}{1+e^{-z}}$ | $\sigma'(z) = \sigma(z)(1 - \sigma(z))$ |
| ReLU | $\max(0, z)$ | $\mathbf{1}[z > 0]$ |

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_grad(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return (z > 0).astype(float)

# Visualização
z = np.linspace(-5, 5, 300)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(z, sigmoid(z),      label='σ(z)',  color='steelblue')
axes[0].plot(z, sigmoid_grad(z), label="σ'(z)", color='steelblue', linestyle='--')
axes[0].set_title('Sigmoide'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(z, relu(z),      label='ReLU(z)',  color='tomato')
axes[1].plot(z, relu_grad(z), label="ReLU'(z)", color='tomato', linestyle='--')
axes[1].set_title('ReLU'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Inicialização dos Pesos

Usamos **inicialização de He** (recomendada para ReLU): $W \sim \mathscr{N}\left(0,\, \sqrt{\frac{2}{n_{in}}}\right)$

Inicializar com zeros causaria **simetria** — todos os neurônios aprenderiam o mesmo. Pesos aleatórios quebram essa simetria.

In [ ]:
def init_params(layer_dims):
    """
    layer_dims: lista com os tamanhos de cada camada, ex: [29, 16, 1]
    Retorna dicionário com W1, b1, W2, b2, ...
    """
    params = {}
    L = len(layer_dims) - 1
    for l in range(1, L + 1):
        n_in_l  = layer_dims[l - 1]
        n_out_l = layer_dims[l]
        # He initialization
        params[f'W{l}'] = np.random.randn(n_in_l, n_out_l) * np.sqrt(2.0 / n_in_l)
        params[f'b{l}'] = np.zeros((1, n_out_l))
    return params

# Arquitetura: 29 entradas → 16 ocultos (ReLU) → 1 saída (sigmoide)
layer_dims = [n_in, 16, 1]
params = init_params(layer_dims)

for k, v in params.items():
    print(f"{k}: {v.shape}")

## 4. Forward Pass

Para cada camada $l$:

$$Z^{[l]} = A^{[l-1]} W^{[l]} + b^{[l]}$$
$$A^{[l]} = g^{[l]}\left(Z^{[l]}\right)$$

Guardamos os valores intermediários $Z$ em um **cache** — eles serão reutilizados no backprop.

In [ ]:
def forward(X, params):
    """
    Executa o forward pass.
    Retorna:
      y_hat  — predições (n_samples, 1)
      cache  — dict com Z1, A1, Z2, A2 para uso no backprop
    """
    W1, b1 = params['W1'], params['b1']
    W2, b2 = params['W2'], params['b2']

    # Camada 1: ReLU
    Z1 = X @ W1 + b1       # (n, 16)
    A1 = relu(Z1)           # (n, 16)

    # Camada 2: Sigmoide
    Z2 = A1 @ W2 + b2      # (n, 1)
    A2 = sigmoid(Z2)        # (n, 1)  ← predição final

    cache = {'Z1': Z1, 'A1': A1, 'Z2': Z2, 'A2': A2, 'X': X}
    return A2, cache

# Teste rápido
y_hat_test, _ = forward(X_tr[:5], params)
print("Predições (antes de treinar):", y_hat_test.ravel().round(3))

## 5. Função de Perda (Binary Cross-Entropy)

$$\mathscr{L} = -\frac{1}{n}\sum_{i=1}^{n}\left[y^{(i)}\log\hat{y}^{(i)} + (1-y^{(i)})\log(1-\hat{y}^{(i)})\right]$$

In [ ]:
def bce_loss(y_hat, y):
    eps = 1e-12   # evita log(0)
    y_hat = np.clip(y_hat, eps, 1 - eps)
    return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

def accuracy(y_hat, y):
    return np.mean((y_hat >= 0.5) == y)

loss0 = bce_loss(*forward(X_tr, params)[:1], y_tr)
print(f"Loss inicial (sem treino): {loss0:.4f}  — esperado ≈ ln(2) ≈ 0.693")

## 6. Backpropagation

Aplicamos a **regra da cadeia** de trás para frente, camada por camada.

**Camada de saída** (sigmoide + BCE):
$$\delta^{[2]} = \hat{y} - y \quad \text{(gradiente combinado BCE + sigmoide)}$$
$$\frac{\partial \mathscr{L}}{\partial W^{[2]}} = \frac{1}{n}\, A^{[1]\top} \delta^{[2]}$$

**Camada oculta** (ReLU):
$$\delta^{[1]} = \left(\delta^{[2]} W^{[2]\top}\right) \odot \text{ReLU}'(Z^{[1]})$$
$$\frac{\partial \mathscr{L}}{\partial W^{[1]}} = \frac{1}{n}\, X^\top \delta^{[1]}$$

In [ ]:
def backward(y, cache, params):
    """
    Executa o backward pass e retorna os gradientes.
    """
    X  = cache['X']
    Z1 = cache['Z1']
    A1 = cache['A1']
    A2 = cache['A2']
    W2 = params['W2']
    n  = X.shape[0]

    # ── Camada 2 ─────────────────────────────────────────
    # Gradiente combinado da BCE e da sigmoide: ∂L/∂Z2 = ŷ - y
    dZ2 = (A2 - y) / n           # (n, 1)
    dW2 = A1.T @ dZ2              # (16, 1)
    db2 = dZ2.sum(axis=0, keepdims=True)  # (1, 1)

    # ── Camada 1 ─────────────────────────────────────────
    # Propaga o erro: δ1 = (δ2 · W2ᵀ) ⊙ ReLU'(Z1)
    dA1 = dZ2 @ W2.T              # (n, 16)
    dZ1 = dA1 * relu_grad(Z1)     # (n, 16)
    dW1 = X.T @ dZ1               # (n_in, 16)
    db1 = dZ1.sum(axis=0, keepdims=True)  # (1, 16)

    return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}

# Teste de dimensões
y_hat0, cache0 = forward(X_tr, params)
grads0 = backward(y_tr, cache0, params)
for k, v in grads0.items():
    print(f"{k}: {v.shape}  (deve bater com {params[k[1:]].shape})")

## 7. Laço de Treino (Gradient Descent)

Atualização de pesos:

$$W^{[l]} \leftarrow W^{[l]} - \alpha \frac{\partial \mathscr{L}}{\partial W^{[l]}}$$

Implementamos também **early stopping** monitorando a loss de validação.

In [ ]:
def update_params(params, grads, lr):
    """Gradient descent: W ← W - α·∂L/∂W"""
    new_params = {}
    for key in params:
        new_params[key] = params[key] - lr * grads['d' + key]
    return new_params


def train(X_tr, y_tr, X_val, y_val,
          layer_dims, lr=0.01, epochs=1000, patience=20):

    params    = init_params(layer_dims)
    history   = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_loss = float('inf')
    best_params = None
    wait = 0

    for epoch in range(epochs):
        # Forward + backward
        y_hat, cache = forward(X_tr, params)
        grads        = backward(y_tr, cache, params)
        params       = update_params(params, grads, lr)

        # Métricas
        train_loss = bce_loss(y_hat, y_tr)
        val_hat, _ = forward(X_val, params)
        val_loss   = bce_loss(val_hat, y_val)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(accuracy(y_hat, y_tr))
        history['val_acc'].append(accuracy(val_hat, y_val))

        # Early stopping
        if val_loss < best_loss:
            best_loss   = val_loss
            best_params = {k: v.copy() for k, v in params.items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping na época {epoch + 1}")
                break

    return best_params, history

In [ ]:
best_params, history = train(
    X_tr, y_tr, X_val, y_val,
    layer_dims=[n_in, 16, 1],
    lr=0.05,
    epochs=2000,
    patience=30,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Treino')
axes[0].plot(history['val_loss'],   label='Validação')
axes[0].set_title('Loss (BCE)')
axes[0].set_xlabel('Época')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['train_acc'], label='Treino')
axes[1].plot(history['val_acc'],   label='Validação')
axes[1].set_title('Acurácia')
axes[1].set_xlabel('Época')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Avaliação

In [ ]:
y_hat_te, _ = forward(X_te, best_params)
test_loss   = bce_loss(y_hat_te, y_te)
test_acc    = accuracy(y_hat_te, y_te)

print(f"Loss no teste : {test_loss:.4f}")
print(f"Acurácia no teste: {test_acc:.1%}")

y_pred = (y_hat_te >= 0.5).astype(int).ravel()
print()
print(classification_report(y_te.ravel().astype(int), y_pred,
                             target_names=['Sem doença', 'Com doença']))

ConfusionMatrixDisplay(
    confusion_matrix(y_te.ravel().astype(int), y_pred),
    display_labels=['Sem doença', 'Com doença']
).plot(cmap='Blues')
plt.title('Matriz de Confusão — Conjunto de Teste')
plt.show()

## 9. Visualizando o Grafo Computacional

Diagrama simplificado do forward e backward para uma rede de 2 camadas, mostrando o fluxo dos gradientes.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.set_xlim(0, 13); ax.set_ylim(0, 5); ax.axis('off')
ax.set_facecolor('#f8fafc')
fig.patch.set_facecolor('#f8fafc')

# Nós
nodes = {
    'X':    (1.0, 2.5),
    'W1':   (1.0, 4.0),
    'b1':   (1.0, 1.0),
    'Z1':   (3.5, 2.5),
    'A1':   (5.5, 2.5),
    'W2':   (5.5, 4.0),
    'b2':   (5.5, 1.0),
    'Z2':   (8.0, 2.5),
    'ŷ':    (10.0, 2.5),
    'L':    (12.0, 2.5),
}

colors = {
    'X': '#dbeafe', 'W1': '#fef9c3', 'b1': '#fef9c3',
    'Z1': '#dcfce7', 'A1': '#dcfce7',
    'W2': '#fef9c3', 'b2': '#fef9c3',
    'Z2': '#dcfce7', 'ŷ': '#dcfce7',
    'L':  '#fee2e2',
}

for name, (x, y) in nodes.items():
    ax.add_patch(plt.Circle((x, y), 0.45, color=colors[name], ec='#475569', lw=1.5, zorder=3))
    ax.text(x, y, name, ha='center', va='center', fontsize=11, fontweight='bold', zorder=4)

# Arestas forward (azul)
forward_edges = [
    ('X', 'Z1'), ('W1', 'Z1'), ('b1', 'Z1'),
    ('Z1', 'A1'),
    ('A1', 'Z2'), ('W2', 'Z2'), ('b2', 'Z2'),
    ('Z2', 'ŷ'), ('ŷ', 'L'),
]
for src, dst in forward_edges:
    x0, y0 = nodes[src]
    x1, y1 = nodes[dst]
    ax.annotate('', xy=(x1 - 0.45 * np.sign(x1-x0), y1 - 0.45 * np.sign(y1-y0)),
                xytext=(x0 + 0.45 * np.sign(x1-x0), y0 + 0.45 * np.sign(y1-y0)),
                arrowprops=dict(arrowstyle='->', color='#2563eb', lw=1.5), zorder=2)

# Seta backward (vermelho, por baixo)
backward_nodes = ['L', 'ŷ', 'Z2', 'A1', 'Z1']
bx = [nodes[n][0] for n in backward_nodes]
for i in range(len(bx) - 1):
    ax.annotate('', xy=(bx[i+1] + 0.45, 0.6),
                xytext=(bx[i] - 0.45, 0.6),
                arrowprops=dict(arrowstyle='->', color='#dc2626', lw=1.5,
                                connectionstyle='arc3,rad=0'), zorder=2)

# Labels das camadas
ax.text(3.5, 4.7, 'Z¹ = X·W¹ + b¹', ha='center', fontsize=9, color='#475569')
ax.text(5.5, 4.7, 'A¹ = ReLU(Z¹)', ha='center', fontsize=9, color='#475569')
ax.text(8.0, 4.7, 'Z² = A¹·W² + b²', ha='center', fontsize=9, color='#475569')
ax.text(10.0, 4.7, 'ŷ = σ(Z²)', ha='center', fontsize=9, color='#475569')
ax.text(12.0, 4.7, 'L = BCE(ŷ,y)', ha='center', fontsize=9, color='#475569')

ax.text(6.5, 0.15, '← backward (regra da cadeia)', ha='center', fontsize=9,
        color='#dc2626', style='italic')

fwd = mpatches.Patch(color='#2563eb', label='Forward pass')
bwd = mpatches.Patch(color='#dc2626', label='Backward pass')
ax.legend(handles=[fwd, bwd], loc='lower right', fontsize=9)

ax.set_title('Grafo Computacional — Rede 2 camadas', fontsize=13, pad=10)
plt.tight_layout()
plt.show()

---

## Resumo

| Componente | O que implementamos |
|---|---|
| `sigmoid`, `relu` | Funções de ativação e suas derivadas |
| `init_params` | Inicialização de He para pesos aleatórios |
| `forward` | Multiplicação de matrizes + ativação, com cache |
| `bce_loss` | Binary Cross-Entropy |
| `backward` | Regra da cadeia camada a camada |
| `update_params` | Gradient descent: $W \leftarrow W - \alpha \nabla W$ |
| `train` | Laço completo + early stopping |

Frameworks como Keras e PyTorch automatizam exatamente estas etapas — mas o mecanismo subjacente é este.